In [1]:
import joblib
import pandas as pd

In [2]:
df = joblib.load("dataset.pkl")
similarity = joblib.load("similarity.pkl")

print(df.shape)

(522, 7)


In [3]:
df.head()

,id,title,description,category,price,is_sold,combined_text
0,1,graphics kit,gently used with no scratches,Others,1000.0,True,graphics kit others gently with no scratches
1,2,Engineering Mathematics 2,2025 version gently used,Books,250.0,False,engineering mathematics 2 books 2025 version g...
2,3,Redgear Mechanical Keyboard,Brand: Redgear\nCondition: Excellent\n\nUsed c...,Electronics,1496.0,False,redgear mechanical keyboard electronics brand ...
3,4,Electronic Component Kit - Like New,Brand: Generic\nCondition: Fair\n\nEverything ...,Stationery and Tools,453.0,False,electronic component kit stationery and tools ...
4,5,Precision Screwdriver Set - Like New,Brand: Taparia\nCondition: Excellent\n\nEveryt...,Stationery and Tools,578.0,True,precision screwdriver set stationery and tools...


In [4]:
id_to_index = pd.Series(
    data=df.index,
    index=df["id"]
)

id_to_index.head()

id
1    0
2    1
3    2
4    3
5    4
dtype: int64

In [5]:
item_id = df.iloc[0]["id"]

print(item_id)

1


In [6]:
print(id_to_index[item_id])

0


In [7]:
item_id = 3

item_index = id_to_index[item_id]

print(item_index)

2


In [8]:
scores = list(enumerate(similarity[item_index]))

scores[:10]

[(0, np.float64(0.0)),
 (1, np.float64(0.0)),
 (2, np.float64(1.0000000000000002)),
 (3, np.float64(0.018310454629700258)),
 (4, np.float64(0.01716899344609269)),
 (5, np.float64(0.02063861189251847)),
 (6, np.float64(0.01615442544721933)),
 (7, np.float64(0.0197362219095058)),
 (8, np.float64(0.019282071446472853)),
 (9, np.float64(0.04913703330285481))]

In [9]:
scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

In [10]:
scores[:10]

[(2, np.float64(1.0000000000000002)),
 (360, np.float64(0.5762174054217671)),
 (196, np.float64(0.5627039737651208)),
 (161, np.float64(0.5144737581486951)),
 (192, np.float64(0.5097298498476898)),
 (51, np.float64(0.5055680036306014)),
 (387, np.float64(0.5055680036306014)),
 (57, np.float64(0.48993687490092047)),
 (230, np.float64(0.48303076925411437)),
 (189, np.float64(0.480886868185052))]

In [11]:
scores = scores[1:]

In [12]:
available = []

for idx, score in scores:

    if not df.iloc[idx]["is_sold"]:

        available.append((idx, score))

In [13]:
top5 = available[:5]

top5

[(360, np.float64(0.5762174054217671)),
 (196, np.float64(0.5627039737651208)),
 (161, np.float64(0.5144737581486951)),
 (192, np.float64(0.5097298498476898)),
 (51, np.float64(0.5055680036306014))]

In [14]:
for idx, score in top5:

    print("Title :", df.iloc[idx]["title"])
    print("Price :", df.iloc[idx]["price"])
    print("Category :", df.iloc[idx]["category"])
    print("Similarity :", round(score,3))
    print("-"*40)

Title : Mi Mi Power Bank 20000mAh for Sale
Price : 1335.0
Category : Electronics
Similarity : 0.576
----------------------------------------
Title : Logitech Logitech G102
Price : 641.0
Category : Electronics
Similarity : 0.563
----------------------------------------
Title : Lenovo ThinkPad E14 in Excellent Condition
Price : 33445.0
Category : Electronics
Similarity : 0.514
----------------------------------------
Title : Dell Inspiron 3511
Price : 22567.0
Category : Electronics
Similarity : 0.51
----------------------------------------
Title : Boat Boat Rockerz 450 for Sale
Price : 1700.0
Category : Electronics
Similarity : 0.506
----------------------------------------


In [15]:
def recommend(item_id, top_n=5):

    if item_id not in id_to_index:
        return pd.DataFrame()

    item_index = id_to_index[item_id]

    scores = list(enumerate(similarity[item_index]))

    scores = sorted(scores,
                    key=lambda x: x[1],
                    reverse=True)

    recommendations = []

    for idx, score in scores[1:]:

        if not df.iloc[idx]["is_sold"]:

            recommendations.append({
                "id": df.iloc[idx]["id"],
                "title": df.iloc[idx]["title"],
                "price": df.iloc[idx]["price"],
                "category": df.iloc[idx]["category"],
                "similarity": round(score,3)
            })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [16]:
recommend(3)

,id,title,price,category,similarity
0,361,Mi Mi Power Bank 20000mAh for Sale,1335.0,Electronics,0.576
1,197,Logitech Logitech G102,641.0,Electronics,0.563
2,162,Lenovo ThinkPad E14 in Excellent Condition,33445.0,Electronics,0.514
3,193,Dell Inspiron 3511,22567.0,Electronics,0.510
4,52,Boat Boat Rockerz 450 for Sale,1700.0,Electronics,0.506
